# 🦴 RSNA BONE AGE ASSESSMENT — MODULAR PIPELINE (NOTEBOOK 02/03)
## ⚡ Module 02: Tri-Model Training Matrix & Full-State Checkpoint Resumption
**Đồ Án Nghiên Cứu Học Thuật Chuẩn DUT — VisionLab Corp (CORP-01-CV)**
* **Mục tiêu**: Huấn luyện chuỗi 3 mô hình đối đầu (ResNet-50 vs ConvNeXt-Tiny/EfficientNet vs Swin-T) trên bộ dữ liệu ảnh $512 \times 512$ đã làm sạch từ Notebook 01.
* **Tính năng Kỹ nghệ Đột phá**:
  1. **Full-State Checkpoint & Auto-Resume**: Tự động lưu vết và khôi phục khi bị đứt kết nối / crash web, tiếp tục huấn luyện ngay tại Epoch dở dang.
  2. **Unified Multimodal Architecture**: Hỗ trợ linh hoạt chuyển đổi giữa 3 trường phái mô hình chỉ bằng 1 dòng cấu hình.
  3. **FP16 Mixed Precision**: Tối ưu hóa bộ nhớ GPU Tesla T4/P100, tăng tốc độ huấn luyện gấp 2 lần.
* **Đầu vào**: Thư mục `/kaggle/input/rsna-boneage-preprocessed-512/` (hoặc output từ Notebook 01).
* **Đầu ra**: Trọng số tối ưu `checkpoint_best.pth`, file lưu vết `checkpoint_last.pth`, và `training_history.csv`.


In [ ]:
# ==============================================================================
# BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG PYTORCH, GPU & SEED TÁI LẬP NGHIỆM
# ==============================================================================
import os
import sys
import time
import math
import random
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# Cố định ngẫu nhiên tái lập nghiệm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"⚡ Device đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"🎮 Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Tổng VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# ==============================================================================
# BƯỚC 2: CẤU HÌNH DỮ LIỆU & BỘ TĂNG CƯỜNG ẢNH Y TẾ AN TOÀN (SAFE AUGMENTATION)
# ==============================================================================
# Xác định đường dẫn dữ liệu đã tiền xử lý
CANDIDATE_DATASET_DIRS = [
    Path('/kaggle/input/rsna-boneage-preprocessed-512'),
    Path('/kaggle/working/rsna_processed_512'),
    Path('./rsna_processed_512')
]

DATASET_DIR = None
for p in CANDIDATE_DATASET_DIRS:
    if (p / 'train_stratified.csv').exists():
        DATASET_DIR = p
        break

if DATASET_DIR is None:
    raise FileNotFoundError("❌ Không tìm thấy bộ dữ liệu đã tiền xử lý từ Notebook 01! Hãy kiểm tra lại Kaggle Dataset.")

CSV_PATH = DATASET_DIR / 'train_stratified.csv'
IMG_DIR = DATASET_DIR / 'images'

df_meta = pd.read_csv(CSV_PATH)
print(f"📁 Thư mục dữ liệu nạp vào: {DATASET_DIR}")
print(f"📊 Thống kê mẫu theo phân vùng:")
print(df_meta['split'].value_counts())

# Định nghĩa Dataset Đa phương thức
class RSNABoneAgeDataset(Dataset):
    """
    PyTorch Dataset đa phương thức:
    - Ảnh X-quang đã qua tiền xử lý (512x512)
    - Giới tính lâm sàng (1.0: Nam, 0.0: Nữ)
    - Nhãn mục tiêu: Tuổi xương (tháng)
    """
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = int(row['id'])
        img_path = self.img_dir / f"{img_id}.png"
        
        # Đọc ảnh mức xám và nhân 3 kênh màu (phù hợp với pre-trained ImageNet)
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        gender = torch.tensor([1.0 if row['male'] else 0.0], dtype=torch.float32)
        boneage = torch.tensor([row['boneage']], dtype=torch.float32)
        
        return img, gender, boneage

# Chuẩn hóa ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Bộ tăng cường dữ liệu y tế an toàn (Medical-Safe Transformations)
# CẤM lật dọc (Vertical Flip) vì giải phẫu bàn tay luôn hướng ngón lên trên
train_transforms = T.Compose([
    T.RandomRotation(degrees=(-10, 10)),                                  # Xoay nhẹ tự nhiên
    T.RandomAffine(degrees=0, translate=(0.04, 0.04), scale=(0.95, 1.05)),# Co giãn & dịch nhẹ
    T.RandomHorizontalFlip(p=0.5),                                        # Lật ngang (Tay trái <-> Tay phải)
    T.ColorJitter(brightness=0.1, contrast=0.1),                          # Biến thiên quang học nhẹ
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Tách DataFrames theo nhãn phân tầng cố định
train_df = df_meta[df_meta['split'] == 'train'].copy()
val_df   = df_meta[df_meta['split'] == 'val'].copy()
test_df  = df_meta[df_meta['split'] == 'test'].copy()

# Khởi tạo DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() or 4

train_loader = DataLoader(RSNABoneAgeDataset(train_df, IMG_DIR, train_transforms), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(RSNABoneAgeDataset(val_df, IMG_DIR, val_test_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(RSNABoneAgeDataset(test_df, IMG_DIR, val_test_transforms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✅ DataLoaders sẵn sàng! Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | Test Batches: {len(test_loader)}")


In [ ]:
# ==============================================================================
# BƯỚC 3: XÂY DỰNG KIẾN TRÚC MÔ HÌNH ĐA PHƯƠNG THỨC ĐA TRƯỜNG PHÁI
# ==============================================================================
class UnifiedMultimodalBoneAge(nn.Module):
    """
    Kiến trúc Deep Learning Đa phương thức tổng quát:
    Hỗ trợ linh hoạt 3 trường phái Backbone:
    1. 'resnet50'        : Residual Bottleneck CNN (2048D)
    2. 'convnext_tiny'   : Modern Pure CNN với Depthwise Separable 7x7 (768D)
    3. 'swin_t'          : Hierarchical Vision Transformer với Shifted Windows (768D)
    """
    def __init__(self, backbone_type="resnet50", gender_dim=32, dropout_rate=0.3, pretrained=True):
        super().__init__()
        self.backbone_type = backbone_type.lower()
        
        # 1. NHÁNH THỊ GIÁC (VISUAL BACKBONE)
        if self.backbone_type == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            base = models.resnet50(weights=weights)
            self.backbone = nn.Sequential(*list(base.children())[:-1]) # Giữ tới AdaptiveAvgPool
            self.vis_dim = 2048
            
        elif self.backbone_type == "convnext_tiny":
            weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
            base = models.convnext_tiny(weights=weights)
            self.backbone = base.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.vis_dim = 768
            
        elif self.backbone_type == "swin_t":
            weights = models.Swin_T_Weights.DEFAULT if pretrained else None
            base = models.swin_t(weights=weights)
            self.backbone = base.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.vis_dim = 768
            
        else:
            raise ValueError(f"Backbone '{backbone_type}' chưa được hỗ trợ. Chọn: resnet50, convnext_tiny, swin_t.")

        # 2. NHÁNH LÂM SÀNG (GENDER EMBEDDING MLP 1D -> 32D)
        self.gender_mlp = nn.Sequential(
            nn.Linear(1, gender_dim),
            nn.BatchNorm1d(gender_dim),
            nn.ReLU(inplace=True),
            nn.Linear(gender_dim, gender_dim),
            nn.BatchNorm1d(gender_dim),
            nn.ReLU(inplace=True)
        )
        
        # 3. ĐẦU HỒI QUY PHÂN TẦNG NÉN DẦN (HIERARCHICAL REGRESSION HEAD)
        combined_dim = self.vis_dim + gender_dim
        self.regression_head = nn.Sequential(
            nn.Linear(combined_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),
            
            nn.Linear(512, 1) # Đầu ra tuyến tính (Tháng tuổi)
        )

    def forward(self, img, gender):
        # Trích xuất đặc trưng ảnh
        if self.backbone_type == "resnet50":
            feat = self.backbone(img) # [B, 2048, 1, 1]
            f_img = torch.flatten(feat, 1)
        elif self.backbone_type == "convnext_tiny":
            feat = self.pool(self.backbone(img))
            f_img = torch.flatten(feat, 1)
        elif self.backbone_type == "swin_t":
            # Swin features shape: [B, H, W, C] -> Permute to [B, C, H, W]
            feat = self.backbone(img).permute(0, 3, 1, 2)
            f_img = torch.flatten(self.pool(feat), 1)

        # Trích xuất đặc trưng giới tính
        e_g = self.gender_mlp(gender) # [B, 32]
        
        # Late Fusion: Ghép nối vector đặc trưng
        z = torch.cat([f_img, e_g], dim=1) # [B, vis_dim + 32]
        
        # Dự đoán tuổi xương
        out = self.regression_head(z) # [B, 1]
        return out

# Kiểm thử khởi tạo mô hình
test_model = UnifiedMultimodalBoneAge(backbone_type="resnet50", gender_dim=32, pretrained=True).to(device)
params_count = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"🏗️ Khởi tạo mô hình kiểm thử thành công! Tổng tham số: {params_count:,}")


In [ ]:
# ==============================================================================
# BƯỚC 4: HỆ THỐNG LƯU VẾT & TỰ ĐỘNG PHỤC HỒI (FULL-STATE CHECKPOINT ENGINE)
# ==============================================================================
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints') if os.path.exists('/kaggle/input') else Path('./checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_full_checkpoint(model, optimizer, scheduler, scaler, epoch, best_val_mae, history, model_name, is_best=False):
    """
    Lưu trữ trạng thái toàn diện của cả 5 đối tượng tối ưu.
    Cho phép khôi phục huấn luyện mượt mà mà không làm vọt learning rate hay bùng nổ gradient.
    """
    checkpoint_state = {
        'epoch': epoch,
        'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'scaler_state_dict': scaler.state_dict() if scaler else None,
        'best_val_mae': best_val_mae,
        'history': history,
        'seed': SEED
    }
    
    last_path = CHECKPOINT_DIR / f"{model_name}_checkpoint_last.pth"
    torch.save(checkpoint_state, last_path)
    
    if is_best:
        best_path = CHECKPOINT_DIR / f"{model_name}_checkpoint_best.pth"
        torch.save(checkpoint_state, best_path)
        print(f"🌟 ĐÃ LƯU CHECKPOINT TỐT NHẤT MỚI: {best_path} (Val MAE: {best_val_mae:.2f} tháng)")

def load_full_checkpoint(model, optimizer, scheduler, scaler, model_name):
    """
    Tự động phát hiện và khôi phục tiến trình từ tệp lưu vết gần nhất.
    """
    last_path = CHECKPOINT_DIR / f"{model_name}_checkpoint_last.pth"
    if last_path.exists():
        print(f"🔄 Tìm thấy tệp lưu vết: {last_path}! Đang khôi phục tiến trình...")
        ckpt = torch.load(last_path, map_location=device)
        
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt['scheduler_state_dict']:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt['scaler_state_dict']:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
            
        start_epoch = ckpt['epoch'] + 1
        best_val_mae = ckpt['best_val_mae']
        history = ckpt['history']
        print(f"✅ Khôi phục thành công! Sẽ tiếp tục huấn luyện từ Epoch [{start_epoch + 1}]...")
        return start_epoch, best_val_mae, history
    else:
        print(f"🚀 Không tìm thấy tệp lưu vết cũ. Bắt đầu huấn luyện mới từ Epoch [1]...")
        return 0, float('inf'), []


In [ ]:
# ==============================================================================
# BƯỚC 5: HÀM HUẤN LUYỆN & KIỂM ĐỊNH VỚI HUBER LOSS & MIXED PRECISION (AMP FP16)
# ==============================================================================
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    running_mae = 0.0
    total_samples = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for imgs, genders, targets in pbar:
        imgs = imgs.to(device, non_blocking=True)
        genders = genders.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Huấn luyện độ chính xác hỗn hợp FP16
        with torch.amp.autocast('cuda'):
            preds = model(imgs, genders)
            loss = criterion(preds, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_mae += torch.sum(torch.abs(preds - targets)).item()
        total_samples += batch_size
        
        pbar.set_postfix({'loss': f"{running_loss/total_samples:.3f}", 'mae': f"{running_mae/total_samples:.2f}m"})
        
    epoch_loss = running_loss / total_samples
    epoch_mae = running_mae / total_samples
    return epoch_loss, epoch_mae

@torch.no_grad()
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_mae = 0.0
    total_samples = 0
    
    for imgs, genders, targets in dataloader:
        imgs = imgs.to(device, non_blocking=True)
        genders = genders.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda'):
            preds = model(imgs, genders)
            loss = criterion(preds, targets)
            
        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        running_mae += torch.sum(torch.abs(preds - targets)).item()
        total_samples += batch_size
        
    epoch_loss = running_loss / total_samples
    epoch_mae = running_mae / total_samples
    return epoch_loss, epoch_mae


In [ ]:
# ==============================================================================
# BƯỚC 6: ĐIỀU PHỐI HUẤN LUYỆN MÔ HÌNH (THIẾT LẬP THAM SỐ VÀ CHẠY THỰC TẾ)
# ==============================================================================
# LỰA CHỌN MÔ HÌNH HUẤN LUYỆN: 'resnet50' | 'convnext_tiny' | 'swin_t'
ACTIVE_BACKBONE = "resnet50"
TOTAL_EPOCHS = 15
LEARNING_RATE = 1e-4

print(f"🎯 BẮT ĐẦU CẤU HÌNH HUẤN LUYỆN CHO: {ACTIVE_BACKBONE.upper()}")

# Khởi tạo mô hình
model = UnifiedMultimodalBoneAge(backbone_type=ACTIVE_BACKBONE, gender_dim=32, pretrained=True).to(device)

# Hàm mất mát Smooth L1 (Huber Loss, delta=1.0)
criterion = nn.SmoothL1Loss(beta=1.0)

# Bộ tối ưu AdamW
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Điều chỉnh tốc độ học Cosine Annealing
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

# Bộ cân chỉnh FP16
scaler = torch.amp.GradScaler('cuda')

# Tự động nạp lưu vết nếu có
start_epoch, best_val_mae, history = load_full_checkpoint(model, optimizer, scheduler, scaler, ACTIVE_BACKBONE)

print(f"🚀 Bắt đầu quá trình huấn luyện trên GPU: {torch.cuda.get_device_name(0)}...")

for epoch in range(start_epoch, TOTAL_EPOCHS):
    epoch_start_time = time.time()
    
    train_loss, train_mae = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_mae = validate_epoch(model, val_loader, criterion, device)
    
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    epoch_duration = time.time() - epoch_start_time
    
    is_best = val_mae < best_val_mae
    if is_best:
        best_val_mae = val_mae
        
    # Ghi nhận lịch sử
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_mae': train_mae,
        'val_loss': val_loss,
        'val_mae': val_mae,
        'lr': current_lr,
        'duration_s': epoch_duration
    })
    
    # Lưu Checkpoint toàn diện
    save_full_checkpoint(model, optimizer, scheduler, scaler, epoch, best_val_mae, history, ACTIVE_BACKBONE, is_best=is_best)
    
    best_tag = "🌟 [Best Saved]" if is_best else ""
    print(f"Epoch [{epoch+1:02d}/{TOTAL_EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val MAE: {val_mae:.2f} thg | LR: {current_lr:.2e} | Thời gian: {epoch_duration:.1f}s {best_tag}")

# Xuất nhật ký huấn luyện ra CSV
history_df = pd.DataFrame(history)
history_csv_path = CHECKPOINT_DIR / f"{ACTIVE_BACKBONE}_training_history.csv"
history_df.to_csv(history_csv_path, index=False)
print(f"\n🎉 HOÀN THÀNH HUẤN LUYỆN {ACTIVE_BACKBONE.upper()}!")
print(f"• MAE tốt nhất trên tập Validation: {best_val_mae:.2f} tháng")
print(f"• File lịch sử xuất ra: {history_csv_path}")


In [ ]:
# ==============================================================================
# BƯỚC 7: TRỰC QUAN HÓA ĐƯỜNG CONG HỌC TẬP (LEARNING CURVES)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Đồ thị Hàm Mất Mát (Loss)
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train Huber Loss', color='#1f77b4', marker='o')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Val Huber Loss', color='#ff7f0e', marker='s')
axes[0].set_title(f"1. Động Học Suy Giảm Hàm Mất Mát ({ACTIVE_BACKBONE.upper()})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Huber Loss")
axes[0].legend()

# Đồ thị Sai Số Tuyệt Đối (Val MAE)
axes[1].plot(history_df['epoch'], history_df['val_mae'], label='Validation MAE (tháng)', color='#2ca02c', marker='^')
axes[1].axhline(best_val_mae, color='red', linestyle='--', label=f"Best MAE: {best_val_mae:.2f}m")
axes[1].set_title(f"2. Động Học Hội Tụ Sai Số MAE (Tháng)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (tháng)")
axes[1].legend()

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / f"{ACTIVE_BACKBONE}_learning_curves.png", dpi=300)
plt.show()
